# Instance Analysis and Visualization

In [ ]:
from zipfile import ZipFile

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from tspn_bnb2 import AnnotatedInstance
from tspn_bnb2.misc.paper_style import init_params

init_params()

## Discover All Instance Collections

In [ ]:
instances = list()
with ZipFile("instances_socg.zip", "r") as zip_file:
    for file in zip_file.namelist():
        if file.endswith(".json"):
            instances.append(AnnotatedInstance.model_validate_json(zip_file.open(file).read()))

## Visualize Random Instances from Each Collection

For each collection, we'll randomly select 9 instances and display them in a 3x3 grid.

The `AnnotatedInstance` objects are converted to `Instance` objects for plotting.

In [ ]:
df = {"instance": [], "instance_type": [], "n": []}

for instance in instances:
    instance_type = "tessellation"

    if "geo_information" in instance.meta:
        instance_type = "OSM"
    elif instance.meta["source"] == "random":
        instance_type = "random"

    df["instance"].append(instance)
    df["instance_type"].append(instance_type)
    df["n"].append(len(instance.polygons))

df = pd.DataFrame(df)

In [ ]:
fig, ax = plt.subplots()

sns.countplot(df, x="n", hue="instance_type", ax=ax)

## Summary Statistics

In [ ]:
for collection_name, instances in df.items():
    if len(instances) == 0:
        continue

    print(f"\n{collection_name} Statistics:")
    print(f"  Total instances: {len(instances)}")

    # Count polygons per instance
    polygon_counts = [len(ann_inst.polygons) for _, ann_inst in instances]
    avg_polys = sum(polygon_counts) / len(polygon_counts)
    print(
        f"  Polygons per instance: min={min(polygon_counts)},"
        f" max={max(polygon_counts)}, avg={avg_polys:.1f}"
    )

    # Count instances with annotations
    annotated_count = sum(1 for _, ann_inst in instances if len(ann_inst.annotations) > 0)
    print(f"  Instances with annotations: {annotated_count}")